# Genre Classifier Training

This notebook trains the CNN-based genre classifier that will be used as a frozen auxiliary loss during style transfer training.

**Workflow:**
1. Load mel spectrogram data
2. Create train/validation datasets
3. Train the classifier
4. Evaluate performance
5. Save trained weights for later use

## 1. Import Libraries

In [ ]:
import sys
sys.path.append('..')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

from models.genre_classifier_loss import GenreClassifierLoss

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Create Dataset Class

In [ ]:
class GenreMelDataset(Dataset):
    """Dataset for mel spectrograms with genre labels."""
    
    def __init__(self, data_dir, genre_mapping={'classical': 0, 'rock': 1}):
        """
        Args:
            data_dir: Path to directory containing genre subdirectories
            genre_mapping: Dict mapping genre names to integer labels
        """
        self.data_dir = Path(data_dir)
        self.genre_mapping = genre_mapping
        self.samples = []
        
        # Collect all mel spectrogram files
        for genre_name, genre_id in genre_mapping.items():
            genre_dir = self.data_dir / genre_name
            if genre_dir.exists():
                mel_files = list(genre_dir.glob('*_mel.pt'))
                for mel_file in mel_files:
                    self.samples.append((mel_file, genre_id))
        
        print(f"Found {len(self.samples)} samples")
        for genre_name, genre_id in genre_mapping.items():
            count = sum(1 for _, gid in self.samples if gid == genre_id)
            print(f"  {genre_name}: {count} samples")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        mel_path, genre_id = self.samples[idx]
        
        # Load mel spectrogram
        mel_spec = torch.load(mel_path)
        
        # Ensure correct shape: (1, H, W)
        if mel_spec.dim() == 2:
            mel_spec = mel_spec.unsqueeze(0)
        elif mel_spec.dim() == 3 and mel_spec.shape[0] != 1:
            mel_spec = mel_spec.mean(dim=0, keepdim=True)
        
        return mel_spec, torch.tensor(genre_id, dtype=torch.long)

## 3. Load and Prepare Data

In [ ]:
# Set data directory
DATA_DIR = Path('../data/output')

# Create dataset
dataset = GenreMelDataset(
    data_dir=DATA_DIR,
    genre_mapping={'classical': 0, 'rock_mel': 1}  # Adjust based on your folder names
)

# Split into train/validation (80/20)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

print(f"\nTrain samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

In [ ]:
# Create data loaders
BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0  # Set to 0 on Windows to avoid multiprocessing issues
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

## 4. Visualize Sample Data

In [ ]:
# Visualize a sample batch
sample_mels, sample_labels = next(iter(train_loader))

print(f"Batch mel shape: {sample_mels.shape}")
print(f"Batch labels shape: {sample_labels.shape}")
print(f"Labels in batch: {sample_labels.tolist()}")

# Plot first 4 samples
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for idx, ax in enumerate(axes.flat):
    if idx < len(sample_mels):
        mel = sample_mels[idx, 0].numpy()
        label = sample_labels[idx].item()
        genre_name = 'Non-Rock' if label == 0 else 'Rock'
        
        ax.imshow(mel, aspect='auto', origin='lower', cmap='viridis')
        ax.set_title(f'Genre: {genre_name} (Label: {label})')
        ax.set_xlabel('Time')
        ax.set_ylabel('Mel Frequency')
        
plt.tight_layout()
plt.show()

## 5. Initialize Genre Classifier

In [ ]:
# Get mel dimensions from sample
mel_height = sample_mels.shape[2]
mel_width = sample_mels.shape[3]

print(f"Mel dimensions: {mel_height} x {mel_width}")

# Initialize classifier (unfrozen for training)
classifier = GenreClassifierLoss(
    num_genres=2,
    mel_height=mel_height,
    mel_width=mel_width,
    embedding_dim=128,
    loss_weight=0.5,  # Not used during training
    device=device,
    freeze_on_init=False  # Keep unfrozen for training
)

classifier = classifier.to(device)
print(f"\nClassifier initialized on {device}")
print(f"Total parameters: {sum(p.numel() for p in classifier.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in classifier.parameters() if p.requires_grad):,}")

## 6. Train the Classifier

In [ ]:
# Training configuration
NUM_EPOCHS = 20
LEARNING_RATE = 1e-3
SAVE_PATH = Path('../notebooks/checkpoints/genre_classifier_best.pt')
SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"Training for {NUM_EPOCHS} epochs")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Save path: {SAVE_PATH}")
print("\nStarting training...\n")

In [ ]:
# Train the classifier
classifier.train_classifier(
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    save_path=str(SAVE_PATH)
)

## 7. Evaluate Final Model

In [ ]:
# Load best model
if SAVE_PATH.exists():
    print(f"Loading best model from {SAVE_PATH}")
    classifier.load_pretrained(str(SAVE_PATH))

# Evaluate on validation set
val_accuracy = classifier.evaluate_classifier(val_loader)
print(f"\nFinal Validation Accuracy: {val_accuracy:.2f}%")

## 8. Test Feature Extraction

In [ ]:
# Test feature extraction with a batch
classifier.eval()
test_mels, test_labels = next(iter(val_loader))
test_mels = test_mels.to(device)

with torch.no_grad():
    features = classifier.extract_features(test_mels)
    print(f"Extracted features shape: {features.shape}")
    print(f"Feature statistics:")
    print(f"  Mean: {features.mean().item():.4f}")
    print(f"  Std: {features.std().item():.4f}")
    print(f"  Min: {features.min().item():.4f}")
    print(f"  Max: {features.max().item():.4f}")

## 9. Visualize Feature Space

In [ ]:
# Extract features for entire validation set
all_features = []
all_labels = []

classifier.eval()
with torch.no_grad():
    for mels, labels in val_loader:
        mels = mels.to(device)
        features = classifier.extract_features(mels)
        all_features.append(features.cpu())
        all_labels.append(labels)

all_features = torch.cat(all_features, dim=0).numpy()
all_labels = torch.cat(all_labels, dim=0).numpy()

print(f"Collected {len(all_features)} feature vectors")

In [ ]:
# Use PCA for visualization
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
features_2d = pca.fit_transform(all_features)

plt.figure(figsize=(10, 8))
for label in [0, 1]:
    mask = all_labels == label
    genre_name = 'Non-Rock' if label == 0 else 'Rock'
    plt.scatter(features_2d[mask, 0], features_2d[mask, 1], 
               label=genre_name, alpha=0.6, s=50)

plt.xlabel('First Principal Component')
plt.ylabel('Second Principal Component')
plt.title('Genre Feature Space (PCA Projection)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Explained variance: {pca.explained_variance_ratio_.sum():.2%}")

## 10. Save Final Model Info

In [ ]:
# Print model info for documentation
print("="*60)
print("GENRE CLASSIFIER TRAINING COMPLETE")
print("="*60)
print(f"\nModel Configuration:")
print(f"  - Number of genres: {classifier.num_genres}")
print(f"  - Mel dimensions: {mel_height} x {mel_width}")
print(f"  - Embedding dimension: {classifier.embedding_dim}")
print(f"  - Architecture: CNN (Conv2d + BatchNorm + MaxPool)")
print(f"\nTraining Results:")
print(f"  - Final validation accuracy: {val_accuracy:.2f}%")
print(f"  - Model saved to: {SAVE_PATH}")
print(f"\nModel Status:")
print(f"  - Parameters frozen: {not next(classifier.parameters()).requires_grad}")
print(f"  - Ready for use as auxiliary loss: ✓")
print("\nNext Steps:")
print("  1. Use GenreClassifierLoss in your main training script")
print("  2. Load pretrained weights with: classifier.load_pretrained(path)")
print("  3. Classifier will be frozen and ready for loss computation")
print("="*60)